In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/company_financial_data.csv")

# --- Feature Engineering ---

# Liquidity stress: low current ratio + low cash = danger
df["liquidity_stress"] = df["current_ratio"] * df["cash_ratio"]

# Profitability composite: combining roa + operating margin
df["profit_score"] = (df["roa"] + df["operating_margin"]) / 2

# Leverage danger: high debt with low coverage = risk
df["leverage_danger"] = df["debt_to_equity"] / (df["interest_coverage"] + 0.01)

# Sentiment combo: financial + news sentiment
df["combined_sentiment"] = (df["mda_sentiment"] + df["news_sentiment"]) / 2

# Growth-profitability gap: growth without profit sustainability
df["growth_profit_gap"] = df["revenue_growth"] - df["operating_margin"]

# Efficiency score: asset utilization efficiency
df["efficiency"] = df["asset_turnover"] * df["gross_margin"]

# --- Output ---
print("New features added:")

new_feats = [
    "liquidity_stress",
    "profit_score",
    "leverage_danger",
    "combined_sentiment",
    "growth_profit_gap",
    "efficiency"
]

print(df[new_feats + ["distressed"]].head(8).round(4))

df.to_csv("data/company_engineered.csv", index=False)

print(f"\nSaved: {df.shape[1]} total features")

New features added:
   liquidity_stress  profit_score  leverage_danger  combined_sentiment  \
0            0.5659        0.1250           0.3844             -0.1025   
1            0.9475        0.0589           0.4742             -0.0330   
2            0.3676       -0.1508           1.6520              0.0180   
3            0.7881        0.0382           0.7755             -0.0110   
4            0.4504        0.0095           0.2751             -0.1500   
5            0.4637       -0.0612           0.3341              0.0755   
6            1.5089        0.1432           0.2657              0.2055   
7            1.0557        0.1820           0.1560              0.3395   

   growth_profit_gap  efficiency  distressed  
0            -0.1435      0.1944           0  
1             0.0674      0.2516           1  
2             0.0996      0.1044           1  
3            -0.2179      0.1497           0  
4             0.1153      0.3582           0  
5             0.1148      0.227

In [2]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

df = pd.read_csv("data/company_engineered.csv")

# Encode sector (text → numbers)
le = LabelEncoder()
df["sector_encoded"] = le.fit_transform(df["sector"])

# Final feature list for the model
feature_cols = [
    "current_ratio","debt_to_equity","roa","roe",
    "gross_margin","operating_margin","interest_coverage",
    "asset_turnover","revenue_growth","cash_ratio",
    "altman_z_score","mda_sentiment","news_sentiment",
    "days_payable_outstanding","liquidity_stress","profit_score",
    "leverage_danger","combined_sentiment","growth_profit_gap",
    "efficiency","sector_encoded","year"
]

X = df[feature_cols]
y = df["distressed"]

print("Features shape:", X.shape)

print("\nTarget distribution:\n", y.value_counts())

print("\nNo missing values:", X.isnull().sum().sum() == 0)

df[feature_cols + ["distressed", "company", "sector"]].to_csv(
    "data/ml_ready.csv", index=False
)

print("ML-ready dataset saved.")

Features shape: (350, 22)

Target distribution:
 distressed
0    221
1    129
Name: count, dtype: int64

No missing values: True
ML-ready dataset saved.
